<a href="https://colab.research.google.com/github/sydneylaub/rush-sales-analysis/blob/analysis-summary/RUSH_Case_Study_GB885_Final_Project_Laub_S.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RUSH Sportswear — US Sales Analysis

**Analyst:** Sydney Laub

**Prepared for:** VP of US Sales

This notebook analyzes RUSH retail sales data from 2020–2021 to answer four
questions from the VP of US Sales and to surface additional trends relevant
to growth planning.

The source data arrives as three raw tables — sales transactions, retailer
locations, and product definitions — and requires cleaning before analysis.

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings for readability
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

## Setup and Data Load

The source data lives in three tables:

- **`TABLE_SALES_885.csv`** — one row per order, with units, price, margin, and sales method
- **`TABLE_RETAILER_885.csv`** — retailer locations, one row per retailer-location
- **`TABLE_PRODUCTS_885.csv`** — product category names

All three are read directly from this repository so the notebook runs
without any local setup.

In [ ]:
# Base URL for the raw data files in this repository
BASE_URL = ('https://raw.githubusercontent.com/sydneylaub/'
            'rush-sales-analysis/refs/heads/main/')

# Load each table. Products is pipe-delimited; all files have a UTF-8 BOM.
sales = pd.read_csv(BASE_URL + 'TABLE_SALES_885.csv', encoding='utf-8-sig')
retailers = pd.read_csv(BASE_URL + 'TABLE_RETAILER_885.csv', encoding='utf-8-sig')
products = pd.read_csv(BASE_URL + 'TABLE_PRODUCTS_885.csv',
                       sep='|', encoding='utf-8-sig')

print(f"Sales:     {sales.shape[0]:,} rows × {sales.shape[1]} columns")
print(f"Retailers: {retailers.shape[0]:,} rows × {retailers.shape[1]} columns")
print(f"Products:  {products.shape[0]:,} rows × {products.shape[1]} columns")

## Initial Inspection

Before cleaning, we inspect each table's structure, data types, and value
ranges to identify quality issues. The data is documented as raw and
unvalidated, so we check for type mismatches, missing values, duplicate
keys, inconsistent categories, and outliers.

In [ ]:
# Structure and data types of the sales table
sales.info()
sales.head()

In [ ]:
# Check for missing values and duplicate orders
print("Missing values by column:")
print(sales.isna().sum()[lambda x: x > 0])
print(f"\nDuplicate ORDER_IDs: {sales['ORDER_ID'].duplicated().sum()}")
print(f"Duplicate full rows:  {sales.duplicated().sum()}")

In [ ]:
# Inspect categorical fields for inconsistent values
print("Sales methods:")
print(sales['SALES_METHOD'].value_counts())
print("\nOrders by year:")
print(sales['YEAR'].value_counts())

In [ ]:
# UNITS_SOLD is stored as text -- identify the non-numeric values
non_numeric = sales[pd.to_numeric(sales['UNITS_SOLD'], errors='coerce').isna()]
print(f"Rows with non-numeric UNITS_SOLD: {len(non_numeric)}")
print(non_numeric[['ORDER_ID', 'RETAILER_ID', 'INVOICE_DATE', 'UNITS_SOLD']])

# Check the numeric ranges for implausible values
print("\nPrice per unit:")
print(sales['PRICE_PER_UNIT'].describe())

In [ ]:
# The retailer table's primary key should be unique -- verify
print(f"Retailer rows:        {len(retailers)}")
print(f"Unique RETAILER_IDs:  {retailers['RETAILER_ID'].nunique()}")

duplicate_ids = retailers[retailers['RETAILER_ID'].duplicated(keep=False)]
print(f"\nRows with duplicated RETAILER_ID: {len(duplicate_ids)}")
print(duplicate_ids.sort_values('RETAILER_ID'))

# Confirm every sales record maps to a known retailer
orphans = set(sales['RETAILER_ID']) - set(retailers['RETAILER_ID'])
print(f"\nRETAILER_IDs in sales with no match: {orphans}")

In [ ]:
# Demonstrate the impact: a naive merge duplicates rows on the collided keys
naive_merge = sales.merge(retailers, on='RETAILER_ID', how='inner')

print(f"Sales rows before merge: {len(sales):,}")
print(f"Rows after naive merge:  {len(naive_merge):,}")
print(f"Rows created by duplicate keys: {len(naive_merge) - len(sales):,}")

### Summary of Issues Found

| # | Issue | Table | Scope |
|---|-------|-------|-------|
| 1 | `PRODUCT_ID` file is pipe-delimited, not comma | Products | Whole file |
| 2 | UTF-8 byte-order mark corrupts first column name | All three | Whole file |
| 3 | `UNITS_SOLD` stored as text due to `***` placeholder | Sales | 2 rows |
| 4 | `PRICE_PER_UNIT` sentinel value of 99999 | Sales | 1 row |
| 5 | `PRICE_PER_UNIT` missing | Sales | 2 rows |
| 6 | Orders with zero units sold | Sales | 4 rows |
| 7 | `SALES_METHOD` misspelled as "Ootlet" | Sales | 20 rows |
| 8 | `RETAILER_ID` not unique — 110 rows, 106 unique IDs | Retailer | 4 IDs, 8 rows |
| 9 | Orphan `RETAILER_ID` 999999999 with no matching retailer | Sales | 1 row |

Issue 8 is the most consequential. `RETAILER_ID` is documented as the primary
key of the retailer table, but four IDs appear twice. The identifier encodes
retailer, region, state, and city — so Walmart and West Gear, which share an
initial, collide at locations they both operate. Joining without addressing
this will duplicate sales records and inflate every revenue figure.

## Data Cleaning

Each issue identified above is addressed below, one at a time, with the
reasoning recorded. Cleaning is performed on copies so the raw tables
remain available for comparison.

Rows are removed only when a required value is unusable and cannot be
responsibly inferred. Every removal is counted and reported.

In [ ]:
# Work on copies so the raw data stays intact for comparison
sales_clean = sales.copy()
retailers_clean = retailers.copy()

rows_start = len(sales_clean)

In [ ]:
# Issue 7: correct the misspelled sales method
sales_clean['SALES_METHOD'] = sales_clean['SALES_METHOD'].replace('Ootlet', 'Outlet')

print(sales_clean['SALES_METHOD'].value_counts())

In [ ]:
# Issue 3: convert UNITS_SOLD to numeric; '***' becomes NaN
sales_clean['UNITS_SOLD'] = pd.to_numeric(sales_clean['UNITS_SOLD'], errors='coerce')

# Issue 4: 99999 is a placeholder, not a real price -- median price is $45
sales_clean.loc[sales_clean['PRICE_PER_UNIT'] == 99999, 'PRICE_PER_UNIT'] = np.nan

# Convert the invoice date from text to datetime
sales_clean['INVOICE_DATE'] = pd.to_datetime(sales_clean['INVOICE_DATE'],
                                             format='%m/%d/%Y')

sales_clean[['PRICE_PER_UNIT', 'UNITS_SOLD', 'INVOICE_DATE']].info()

In [ ]:
# Issues 3, 5, 6: remove rows with unusable price or unit values.
before = len(sales_clean)

sales_clean = sales_clean.dropna(subset=['PRICE_PER_UNIT', 'UNITS_SOLD'])
sales_clean = sales_clean[sales_clean['UNITS_SOLD'] > 0]

print(f"Rows removed: {before - len(sales_clean)} of {before} "
      f"({(before - len(sales_clean)) / before:.2%})")
print(f"Rows remaining: {len(sales_clean):,}")

### Issue 8: Duplicate Retailer IDs

Four `RETAILER_ID` values map to two different retailer records each. In
three cases the conflict is between Walmart and West Gear at the same city;
in one case it is Sports Direct across two different states.

Because `RETAILER_ID` is the only link between a sale and its location,
there is no way to determine from this data which of the two records a
given order belongs to. The affected orders cannot be attributed with
confidence.

We keep one record per ID so the join does not duplicate sales, and we flag
the affected IDs so their impact can be measured. **This is a data
governance issue that should be escalated** — the identifier scheme cannot
distinguish retailers that share an initial and operate in the same city.

In [ ]:
# Record which IDs are ambiguous before resolving them
ambiguous_ids = retailers_clean.loc[
    retailers_clean['RETAILER_ID'].duplicated(keep=False), 'RETAILER_ID'
].unique()

# Keep one record per ID so the join cannot duplicate sales rows
retailers_clean = retailers_clean.drop_duplicates(subset='RETAILER_ID', keep='first')

print(f"Retailer rows: {len(retailers)} -> {len(retailers_clean)}")
print(f"IDs now unique: {retailers_clean['RETAILER_ID'].is_unique}")

# Measure how much of the data is affected by the ambiguity
affected = sales_clean[sales_clean['RETAILER_ID'].isin(ambiguous_ids)]
print(f"\nOrders on ambiguous IDs: {len(affected):,} "
      f"({len(affected) / len(sales_clean):.1%} of orders)")
print(f"Units on ambiguous IDs:  {affected['UNITS_SOLD'].sum():,.0f}")

## Building the Analysis Dataset

With the tables cleaned, we join them into a single dataset. Left joins are
used from the sales table so that any order failing to match is visible
rather than silently dropped.

The data dictionary defines no revenue field, so we derive it as
`PRICE_PER_UNIT × UNITS_SOLD`.

In [ ]:
# Join sales to retailer locations and product names
df = (sales_clean
      .merge(retailers_clean, on='RETAILER_ID', how='left')
      .merge(products, on='PRODUCT_ID', how='left'))

# Verify the join did not create or lose rows unexpectedly
print(f"Sales rows in:  {len(sales_clean):,}")
print(f"Rows after join: {len(df):,}")
print(f"Unmatched retailers: {df['RETAILER'].isna().sum()}")
print(f"Unmatched products:  {df['PRODUCT_NAME'].isna().sum()}")

In [ ]:
# Issue 9: remove the single order whose RETAILER_ID matches no retailer.
# Without a location this order cannot be attributed to a state or retailer.
df = df.dropna(subset=['RETAILER'])

# Derive revenue -- the source data has no total sales column
df['REVENUE'] = df['PRICE_PER_UNIT'] * df['UNITS_SOLD']

print(f"Final analysis dataset: {len(df):,} rows")
print(f"Date range: {df['INVOICE_DATE'].min():%b %Y} to {df['INVOICE_DATE'].max():%b %Y}")
print(f"Total revenue: ${df['REVENUE'].sum():,.0f}")

## Business Questions

The VP of US Sales asked four specific questions. Each is answered below
using the cleaned dataset.

In [ ]:
# Split by year for the year-specific questions
sales_2021 = df[df['YEAR'] == 2021]
sales_2020 = df[df['YEAR'] == 2020]

print(f"2021 orders: {len(sales_2021):,}")
print(f"2020 orders: {len(sales_2020):,}")

In [ ]:
# Q1: Which product category had the highest 2021 sales in dollars?
product_revenue_2021 = (sales_2021
                        .groupby('PRODUCT_NAME')['REVENUE']
                        .sum()
                        .sort_values(ascending=False))

print("2021 revenue by product category:")
print(product_revenue_2021.apply(lambda x: f"${x:,.0f}"))

top_product = product_revenue_2021.index[0]
print(f"\nAnswer: {top_product} — ${product_revenue_2021.iloc[0]:,.0f}")

In [ ]:
# Q2 and Q3: Which state had the highest 2021 sales of women's and men's products?
for gender in ["Women's", "Men's"]:
    state_revenue = (sales_2021[sales_2021['PRODUCT_NAME'].str.startswith(gender)]
                     .groupby('STATE')['REVENUE']
                     .sum()
                     .sort_values(ascending=False))

    print(f"\nTop 5 states — {gender} products, 2021:")
    print(state_revenue.head().apply(lambda x: f"${x:,.0f}"))
    print(f"Answer: {state_revenue.index[0]} — ${state_revenue.iloc[0]:,.0f}")

In [ ]:
# Q4: Which retailer purchased the most units, in 2021 and in 2020?
for year, year_df in [(2021, sales_2021), (2020, sales_2020)]:
    units = (year_df.groupby('RETAILER')['UNITS_SOLD']
             .sum()
             .sort_values(ascending=False))

    print(f"\nUnits sold by retailer, {year}:")
    print(units.apply(lambda x: f"{x:,.0f}"))
    print(f"Answer: {units.index[0]} — {units.iloc[0]:,.0f} units")

## Exploratory Analysis

Beyond the VP's specific questions, three areas were examined for trends
relevant to growth planning: data coverage across years, seasonality and
sales channel performance, and retailer concentration.

In [ ]:
# The two years differ sharply in order volume -- check whether 2020
# coverage is complete before making any year-over-year comparison.
coverage = df.groupby(['YEAR', df['INVOICE_DATE'].dt.month]).size().unstack(fill_value=0)
coverage.columns.name = 'Month'
print("Orders per month by year:")
print(coverage)

print("\nRetailers active by year:")
print(df.groupby('YEAR')['RETAILER'].nunique())
print("\nStates covered by year:")
print(df.groupby('YEAR')['STATE'].nunique())

In [ ]:
# Identify which retailers and regions entered between 2020 and 2021
retailers_2020 = set(df[df['YEAR'] == 2020]['RETAILER'])
retailers_2021 = set(df[df['YEAR'] == 2021]['RETAILER'])

print(f"Active in 2020: {sorted(retailers_2020)}")
print(f"Active in 2021: {sorted(retailers_2021)}")
print(f"New in 2021:    {sorted(retailers_2021 - retailers_2020)}")

# Revenue comparison, with the expansion caveat in view
print("\nRevenue by year:")
print(df.groupby('YEAR')['REVENUE'].sum().apply(lambda x: f"${x:,.0f}"))

# Same-retailer comparison: only those present in both years
both_years = retailers_2020 & retailers_2021
comparable = df[df['RETAILER'].isin(both_years)]
print(f"\nLike-for-like revenue ({len(both_years)} retailers in both years):")
print(comparable.groupby('YEAR')['REVENUE'].sum().apply(lambda x: f"${x:,.0f}"))

### Finding 1: Growth Is Driven by Distribution Expansion

Total revenue grew from $24.2M in 2020 to $95.9M in 2021 — roughly four-fold. That
figure is misleading on its own.

RUSH added two retail partners between 2020 and 2021, Foot Locker and
Walmart, and expanded from 22 states to all 50. Restricting the comparison
to the four retailers present in both years gives like-for-like revenue of
$24.2M to $40.0M — **65% growth in the existing business**.

The remaining $55.9M, roughly 78% oftotal growth, came from the two new
partners. Both figures matter: the existing base is growing healthily, but
the headline number reflects a one-time footprint expansion that will not
repeat in 2022 unless further partners are added.

### Finding 2: Seasonality

In [ ]:
# Monthly revenue by year. 2020 covers fewer retailers and states, so the
# two years are plotted separately rather than combined.
monthly = (df.groupby(['YEAR', df['INVOICE_DATE'].dt.month])['REVENUE']
           .sum()
           .unstack(level=0))
monthly.index.name = 'Month'

print("Monthly revenue ($M):")
print((monthly / 1_000_000).round(2))

In [ ]:
# Plot both years to compare shape rather than absolute level
fig, ax = plt.subplots(figsize=(10, 5))

month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

for year in [2020, 2021]:
    ax.plot(monthly.index, monthly[year] / 1_000_000,
            marker='o', label=str(year))

ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_labels)
ax.set_ylabel('Revenue ($M)')
ax.set_title('Monthly Revenue by Year')
ax.legend(title='Year')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

Monthly revenue in 2021 shows a pronounced seasonal pattern: peaks in July
($10.4M) and December ($10.3M), and a trough in March ($5.2M). The gap
between best and worst month is nearly two-fold.

2020 shows no comparable pattern, drifting between $1.1M and $3.1M and
declining through the second half of the year.

The two shapes are not directly comparable. Foot Locker and Walmart were
added in 2021, so the 2021 curve partly reflects their seasonality rather
than a change in RUSH's own demand pattern. A second full year at the
current footprint would be needed to confirm the July and December peaks
as a durable cycle.

**Implication for planning:** if the pattern holds, inventory and staffing
should be weighted toward summer and the December holiday period, with
March as the natural window for clearance and maintenance activity.

## Summary of Findings

### Answers to the VP's Questions

| Question | Answer |
|---|---|
| Highest-selling product category, 2021 | Men's Street Footwear — $22.6M |
| Top state for women's products, 2021 | Maine — $2.2M |
| Top state for men's products, 2021 | Delaware — $2.3M |
| Retailer purchasing most units, 2021 | Foot Locker — 1,096,890 units |
| Retailer purchasing most units, 2020 | Amazon — 316,880 units |

The 2020 figure carries an important caveat: Foot Locker and Walmart were
not RUSH partners in 2020, so Amazon led a materially smaller field of four
retailers across 22 states.

### Additional Findings

**Growth is driven by distribution expansion, not existing accounts.**
Revenue grew from $24.2M to $95.9M year over year, but $55.9M of that
increase — roughly 78% — came from two new retail partners. Like-for-like
growth among retailers present in both years was 65%. Both numbers are
healthy; reporting only the headline would set an expectation for 2022 that
the existing business cannot meet without further expansion.

**Revenue in 2021 was strongly seasonal**, peaking in July and December and
troughing in March, with a near two-fold swing between best and worst month.

### Data Quality Note

Four `RETAILER_ID` values in the retailer table map to two different
retailer records each, affecting 623 orders and 54,248 units. The
identifier encodes retailer, region, state, and city, so retailers sharing
an initial collide at shared locations — chiefly Walmart and West Gear.

These orders cannot be reliably attributed to a specific retailer. The
analysis contains the problem by retaining one record per ID, but the
underlying identifier scheme should be revised. Joining this data without
addressing the issue inflates the record count by 622 rows and overstates
revenue accordingly.